# AI Engineering: Fine-Tuning with LoRA (Low-Rank Adaptation)

## Efficient Adaptation of Large Language Models

**Problem**: Fine-tuning LLMs requires updating billions of parameters  
**Solution**: Update only low-rank matrices (LoRA) → 100x less memory  
**Innovation**: Hu et al. (2021) showed fine-tuned models have low intrinsic dimension


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd
import pandas as pd

np.random.seed(42)

## Part 1: LoRA Theory

### Standard Fine-Tuning
```
W_new = W + ΔW  [update full matrix, requires O(d²) memory]
```

### LoRA
```
W_new = W + BA  [B: d×r, A: r×d, r << d]
Memory: O(dr) instead of O(d²)  [100x savings for r=100, d=10k]
```

### Convergence Analysis (Supervised Fine-Tuning)
```
L(θ) = (1/n) ∑ -log p(y_i | x_i; θ)  [negative log-likelihood]

Convergence Rate (convex):
L(θ_T) - L(θ*) = O(1/T)

Non-convex (realistic):
||∇L(θ_T)||² = O(1/T)  [converges to critical point]
```


In [ ]:
class LoRAAdapter:
    """Low-Rank Adaptation for fine-tuning."""
    
    def __init__(self, d, r, learning_rate=0.001):
        """Initialize LoRA adapter.
        
        d: dimension of weight matrix
        r: rank of low-rank matrices  (r << d)
        """
        self.d = d
        self.r = r
        self.lr = learning_rate
        
        # Initialize low-rank matrices
        self.A = np.random.randn(r, d) * 0.01  # Small random init
        self.B = np.zeros((d, r))  # B initialized to zero
        
        # Full weight matrix (mock pre-trained model)
        self.W = np.random.randn(d, d) * 0.1
    
    def get_updated_weights(self):
        """Compute W_new = W + BA."""
        return self.W + self.B @ self.A
    
    def compute_gradient(self, batch_loss):
        """Compute gradients for B and A."""
        # Mock gradient (in reality, computed by autograd)
        dB = np.random.randn(*self.B.shape) * batch_loss
        dA = np.random.randn(*self.A.shape) * batch_loss
        return dB, dA
    
    def update_step(self, dB, dA):
        """Update low-rank matrices."""
        self.B -= self.lr * dB
        self.A -= self.lr * dA
    
    def memory_analysis(self):
        """Compare memory usage."""
        full_update = self.d * self.d * 4  # 4 bytes per float32
        lora_update = (self.d * self.r + self.r * self.d) * 4
        savings = (1 - lora_update / full_update) * 100
        return full_update, lora_update, savings

# Test LoRA
adapter = LoRAAdapter(d=10000, r=100)
full_mem, lora_mem, savings = adapter.memory_analysis()

print("LoRA MEMORY ANALYSIS")
print("="*60)
print(f"Weight matrix dimension: {adapter.d} × {adapter.d}")
print(f"Low-rank dimension: r = {adapter.r}")
print(f"\nMemory usage:")
print(f"  Full fine-tuning: {full_mem / 1e6:.1f} MB")
print(f"  LoRA: {lora_mem / 1e6:.1f} MB")
print(f"  Savings: {savings:.1f}%")

## Part 2: Training Dynamics


In [ ]:
# Simulate training convergence
class TrainingSimulator:
    def __init__(self, method='lora', lr=0.01, n_epochs=50):
        self.method = method
        self.lr = lr
        self.n_epochs = n_epochs
        self.theta = np.random.randn(100) * 0.1  # Random init
        self.losses = []
        self.gradients = []
    
    def run(self):
        """Simulate training with loss function L(θ) = ||θ - θ*||²
        (simple quadratic for illustration)
        """
        theta_star = np.random.randn(100)  # Target
        
        for epoch in range(self.n_epochs):
            # Loss and gradient
            loss = np.sum((self.theta - theta_star) ** 2) / len(self.theta)
            grad = 2 * (self.theta - theta_star) / len(self.theta)
            
            self.losses.append(loss)
            self.gradients.append(np.linalg.norm(grad))
            
            # Update
            if self.method == 'full':
                self.theta -= self.lr * grad
            elif self.method == 'lora':
                # LoRA: update only rank-r subspace
                self.theta -= self.lr * grad * 0.8  # Slightly slower due to rank constraint
        
        return np.array(self.losses), np.array(self.gradients)

# Train both methods
sim_full = TrainingSimulator(method='full', lr=0.05)
loss_full, grad_full = sim_full.run()

sim_lora = TrainingSimulator(method='lora', lr=0.05)
loss_lora, grad_lora = sim_lora.run()

print("\nTRAINING CONVERGENCE")
print("="*60)
print(f"Initial loss (full): {loss_full[0]:.4f}")
print(f"Final loss (full): {loss_full[-1]:.6f}")
print(f"Initial loss (LoRA): {loss_lora[0]:.4f}")
print(f"Final loss (LoRA): {loss_lora[-1]:.6f}")
print(f"\nFinal gradient norm (full): {grad_full[-1]:.6f}")
print(f"Final gradient norm (LoRA): {grad_lora[-1]:.6f}")

## Part 3: Visualization & Evaluation


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Loss curves
ax = axes[0, 0]
ax.semilogy(loss_full, 'o-', linewidth=2, label='Full Fine-Tuning', markersize=4)
ax.semilogy(loss_lora, 's-', linewidth=2, label='LoRA (r=100)', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Training Convergence: Full vs LoRA')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

# Plot 2: Gradient norm
ax = axes[0, 1]
ax.semilogy(grad_full, 'o-', linewidth=2, label='Full', markersize=4)
ax.semilogy(grad_lora, 's-', linewidth=2, label='LoRA', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('||∇L(θ)|| (log scale)')
ax.set_title('Gradient Norm Over Training')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

# Plot 3: Memory-Performance Tradeoff
ax = axes[1, 0]
ranks = [1, 5, 10, 50, 100, 500, 1000]
d = 10000
memories = [(d * r + r * d) / 1e6 for r in ranks]  # MB
accuracies = [0.40, 0.65, 0.78, 0.92, 0.95, 0.97, 0.98]  # Mock accuracies

ax.plot(memories, accuracies, 'o-', linewidth=2, markersize=8, color='steelblue')
for r, mem, acc in zip(ranks[::2], memories[::2], accuracies[::2]):
    ax.annotate(f'r={r}', xy=(mem, acc), xytext=(5, 5), textcoords='offset points', fontsize=9)
ax.set_xlabel('Memory Usage (MB)')
ax.set_ylabel('Fine-Tuning Accuracy')
ax.set_title('LoRA: Rank vs Performance')
ax.grid(True, alpha=0.3)
ax.set_ylim([0.3, 1.0])

# Plot 4: Rank comparison
ax = axes[1, 1]
config_names = ['Full\n(100GB)', 'r=1000\n(200MB)', 'r=100\n(20MB)', 'r=10\n(2MB)']
influences = [1.0, 0.99, 0.95, 0.80]  # How much of the info is captured
colors_rank = ['red', 'blue', 'green', 'orange']

bars = ax.barh(config_names, influences, color=colors_rank, alpha=0.7, edgecolor='black')
ax.set_xlabel('Information Captured (fraction of full model)')
ax.set_title('LoRA: Memory vs Expressiveness')
ax.set_xlim([0, 1.1])
for i, (bar, inf) in enumerate(zip(bars, influences)):
    ax.text(inf + 0.02, bar.get_y() + bar.get_height()/2, f'{inf:.1%}', va='center')

plt.tight_layout()
plt.savefig('SECTION_3_AI_ENGINEERING/lora_finetuning.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nLoRA CONFIGURATION COMPARISON")
print("="*60)
for name, mem, acc in zip(['Full', 'r=1000', 'r=100', 'r=10'], 
                          [100000, 200, 20, 2], 
                          [1.00, 0.99, 0.95, 0.80]):
    print(f"{name:<15} Memory: {mem:>8} MB | Accuracy: {acc:.1%}")

## Key Insights

1. **Low intrinsic dimension**: Fine-tuned updates lie in low-rank subspace
2. **Efficiency**: 100x memory savings with minimal accuracy loss
3. **Convergence**: Similar convergence rate to full fine-tuning
4. **Modularity**: Different LoRA matrices for different tasks
5. **Industry standard**: Widely adopted for efficient LLM adaptation (2023-2026)

### References
- Hu, E. J., et al. (2021). "LoRA: Low-Rank Adaptation of Large Language Models"
